<a href="https://colab.research.google.com/github/dairo123e/tratamiento-audio-python/blob/main/Tratamiento_de_Audio_Dairo_Contreras_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Instalación de librerías


In [ ]:
!pip install librosa soundfile noisereduce

In [ ]:
import os
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt

from scipy.signal import butter, sosfiltfilt
from IPython.display import Audio, display
from google.colab import files


# 2. Carga del audio


En esta sección se cargó la grabación original utilizada en la práctica. El archivo contiene una señal de voz con presencia de ruido ambiental, la cual será caracterizada, preprocesada y tratada mediante Python.

In [ ]:
archivos = files.upload()

In [ ]:
nombre_archivo = list(archivos.keys())[0]

print("Archivo cargado correctamente")
print("Nombre del archivo:", nombre_archivo)
print("Tamaño del archivo:", os.path.getsize(nombre_archivo), "bytes")

In [ ]:
audio, frecuencia_muestreo = librosa.load(
    nombre_archivo,
    sr=None,
    mono=False
)

print("Audio cargado correctamente")
print("Dimensiones de la señal:", audio.shape)
print("Frecuencia de muestreo:", frecuencia_muestreo, "Hz")
print("Tipo de dato:", audio.dtype)

In [ ]:
# Preparar el audio para su reproducción

if audio.ndim > 1:
    audio_escucha = librosa.to_mono(audio)
else:
    audio_escucha = audio.copy()

print("Reproducción del audio original:")
display(Audio(audio_escucha, rate=frecuencia_muestreo))

# 3. Caracterización de la señal

En esta sección se identificaron las principales características técnicas de la señal utilizada en la práctica, incluyendo el formato del archivo, el tipo de señal, la duración, la frecuencia de muestreo, el número de canales, la cantidad de muestras y el tipo de dato. Esta información permitió conocer las condiciones iniciales del audio antes de aplicar las operaciones de preprocesamiento y tratamiento.

In [ ]:
# Identificación de las características del audio original

if audio.ndim == 1:
    numero_canales = 1
    numero_muestras = len(audio)
    tipo_senal = "Mono"
else:
    numero_canales = audio.shape[0]
    numero_muestras = audio.shape[1]

    if numero_canales == 2:
        tipo_senal = "Estéreo"
    else:
        tipo_senal = "Multicanal"

duracion = numero_muestras / frecuencia_muestreo
extension = os.path.splitext(nombre_archivo)[1].upper()

print("CARACTERÍSTICAS DE LA SEÑAL DE AUDIO")
print("-" * 50)
print("Nombre del archivo:", nombre_archivo)
print("Formato del archivo:", extension)
print("Tipo de señal:", tipo_senal)
print("Número de canales:", numero_canales)
print("Número de muestras por canal:", numero_muestras)
print("Frecuencia de muestreo:", frecuencia_muestreo, "Hz")
print("Duración:", round(duracion, 2), "segundos")
print("Tipo de dato:", audio.dtype)

In [ ]:
import pandas as pd

tabla_caracteristicas = pd.DataFrame({
    "Característica": [
        "Nombre del archivo",
        "Formato",
        "Tipo de señal",
        "Número de canales",
        "Muestras por canal",
        "Frecuencia de muestreo",
        "Duración",
        "Tipo de dato"
    ],
    "Resultado": [
        nombre_archivo,
        extension,
        tipo_senal,
        numero_canales,
        numero_muestras,
        f"{frecuencia_muestreo} Hz",
        f"{duracion:.2f} segundos",
        str(audio.dtype)
    ]
})

display(tabla_caracteristicas)

# 4. Preprocesamiento

En esta etapa se preparó la señal para facilitar su tratamiento posterior. El audio original fue convertido de estéreo a mono, con el propósito de trabajar con un único canal y reducir la cantidad de información procesada. Posteriormente, la señal fue remuestreada de 48.000 Hz a 22.050 Hz, una frecuencia adecuada para el análisis de voz. Finalmente, se identificaron los silencios ubicados al comienzo y al final de la grabación, conservando temporalmente el segmento inicial como referencia del ruido ambiental.

In [ ]:
# Conversión de la señal original a mono

if audio.ndim > 1:
    audio_mono = librosa.to_mono(audio)
else:
    audio_mono = audio.copy()

print("CONVERSIÓN A MONO")
print("-" * 40)
print("Dimensiones del audio original:", audio.shape)
print("Dimensiones del audio mono:", audio_mono.shape)
print("Número de canales después de la conversión: 1")

In [ ]:
# Remuestreo de la señal

frecuencia_objetivo = 22050

if frecuencia_muestreo != frecuencia_objetivo:
    audio_remuestreado = librosa.resample(
        audio_mono,
        orig_sr=frecuencia_muestreo,
        target_sr=frecuencia_objetivo
    )
else:
    audio_remuestreado = audio_mono.copy()

frecuencia_procesamiento = frecuencia_objetivo

print("REMUESTREO DE LA SEÑAL")
print("-" * 40)
print("Frecuencia original:", frecuencia_muestreo, "Hz")
print("Frecuencia de procesamiento:", frecuencia_procesamiento, "Hz")
print("Muestras antes del remuestreo:", len(audio_mono))
print("Muestras después del remuestreo:", len(audio_remuestreado))
print(
    "Duración después del remuestreo:",
    round(len(audio_remuestreado) / frecuencia_procesamiento, 2),
    "segundos"
)

In [ ]:
print("Reproducción del audio después de convertirlo a mono y remuestrearlo:")
display(Audio(audio_remuestreado, rate=frecuencia_procesamiento))

In [ ]:
# Identificación y recorte de silencios extremos

audio_sin_silencios, intervalo_conservado = librosa.effects.trim(
    audio_remuestreado,
    top_db=30
)

duracion_antes = len(audio_remuestreado) / frecuencia_procesamiento
duracion_despues = len(audio_sin_silencios) / frecuencia_procesamiento

silencio_eliminado = duracion_antes - duracion_despues

print("ANÁLISIS DE SILENCIOS")
print("-" * 40)
print("Duración antes del recorte:",
      round(duracion_antes, 2), "segundos")
print("Duración después del recorte:",
      round(duracion_despues, 2), "segundos")
print("Silencio total identificado:",
      round(silencio_eliminado, 2), "segundos")
print("Intervalo de muestras conservado:",
      intervalo_conservado)

La forma de onda permite representar la variación de la amplitud de la señal a través del tiempo. Esta visualización facilita la identificación de los segmentos de voz, las pausas, los silencios y los cambios de intensidad presentes en la grabación original.

In [ ]:
# Forma de onda del audio antes del tratamiento

plt.figure(figsize=(14, 5))

librosa.display.waveshow(
    audio_remuestreado,
    sr=frecuencia_procesamiento,
    color="steelblue"
)

plt.title("Forma de onda del audio original")
plt.xlabel("Tiempo (segundos)")
plt.ylabel("Amplitud")
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "forma_onda_original.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


El espectrograma representa la distribución de la energía de la señal en función del tiempo y la frecuencia. Los colores de mayor intensidad indican una mayor concentración de energía, mientras que las regiones oscuras representan componentes de menor intensidad. Esta representación permite observar la voz y la presencia de ruido en diferentes bandas de frecuencia.

In [ ]:
# Cálculo del espectrograma del audio original

stft_original = librosa.stft(audio_remuestreado)

espectrograma_original = librosa.amplitude_to_db(
    np.abs(stft_original),
    ref=np.max
)

plt.figure(figsize=(14, 6))

librosa.display.specshow(
    espectrograma_original,
    sr=frecuencia_procesamiento,
    x_axis="time",
    y_axis="hz",
    cmap="magma"
)

plt.colorbar(format="%+2.0f dB")
plt.title("Espectrograma del audio original")
plt.xlabel("Tiempo (segundos)")
plt.ylabel("Frecuencia (Hz)")
plt.ylim(0, 8000)
plt.tight_layout()

plt.savefig(
    "espectrograma_original.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 5. Tratamiento del audio


Debido a que la grabación contiene aproximadamente un segundo inicial de ruido ambiental antes del comienzo de la voz, este intervalo fue seleccionado como perfil de referencia. La muestra permitió caracterizar el ruido presente y aplicar posteriormente el algoritmo de reducción, procurando conservar la información correspondiente a la voz.

In [ ]:
# Selección del primer segundo como perfil de ruido

inicio_ruido_seg = 0.0
fin_ruido_seg = 1.0

inicio_ruido = int(
    inicio_ruido_seg * frecuencia_procesamiento
)

fin_ruido = int(
    fin_ruido_seg * frecuencia_procesamiento
)

perfil_ruido = audio_remuestreado[
    inicio_ruido:fin_ruido
]

print("PERFIL DE RUIDO SELECCIONADO")
print("-" * 45)
print("Inicio:", inicio_ruido_seg, "segundos")
print("Final:", fin_ruido_seg, "segundos")
print("Duración:", fin_ruido_seg - inicio_ruido_seg, "segundos")
print("Cantidad de muestras:", len(perfil_ruido))


In [ ]:
print("Reproducción del perfil de ruido seleccionado:")

display(
    Audio(
        perfil_ruido,
        rate=frecuencia_procesamiento
    )
)

In [ ]:
plt.figure(figsize=(14, 5))

librosa.display.waveshow(
    audio_remuestreado,
    sr=frecuencia_procesamiento,
    color="steelblue",
    alpha=0.8
)

plt.axvspan(
    inicio_ruido_seg,
    fin_ruido_seg,
    color="red",
    alpha=0.35,
    label="Perfil de ruido"
)

plt.title("Ubicación del perfil de ruido seleccionado")
plt.xlabel("Tiempo (segundos)")
plt.ylabel("Amplitud")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "perfil_ruido_seleccionado.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Reducción del ruido ambiental

audio_reducido = nr.reduce_noise(
    y=audio_remuestreado,
    sr=frecuencia_procesamiento,
    y_noise=perfil_ruido,
    prop_decrease=0.60,
    stationary=False
)

print("REDUCCIÓN DE RUIDO COMPLETADA")
print("-" * 45)
print("Muestras de entrada:", len(audio_remuestreado))
print("Muestras de salida:", len(audio_reducido))
print("Intensidad de reducción:", 0.60)
print("Modo utilizado: ruido variable")

In [ ]:
print("AUDIO ORIGINAL")
display(
    Audio(
        audio_remuestreado,
        rate=frecuencia_procesamiento
    )
)

print("AUDIO CON REDUCCIÓN DE RUIDO")
display(
    Audio(
        audio_reducido,
        rate=frecuencia_procesamiento
    )
)

La reducción inicial no produjo una diferencia auditiva suficiente, debido a que el ruido variaba durante la grabación y el segmento inicial presentaba una intensidad considerable. Por esta razón, se ajustó el procedimiento mediante la eliminación controlada del ruido previo a la voz, una reducción no estacionaria y una atenuación de las regiones de baja energía.

In [ ]:
# Eliminación del ruido fuerte anterior al comienzo de la voz

inicio_voz_seg = 1.0

muestra_inicio_voz = int(
    inicio_voz_seg * frecuencia_procesamiento
)

audio_base = audio_remuestreado[muestra_inicio_voz:]

print("RECORTE DEL RUIDO INICIAL")
print("-" * 45)
print("Inicio de la voz estimado:", inicio_voz_seg, "segundos")
print(
    "Duración original:",
    round(
        len(audio_remuestreado) / frecuencia_procesamiento,
        2
    ),
    "segundos"
)
print(
    "Duración después del recorte:",
    round(
        len(audio_base) / frecuencia_procesamiento,
        2
    ),
    "segundos"
)

display(Audio(audio_base, rate=frecuencia_procesamiento))

In [ ]:
# Reducción de ruido variable sin perfil fijo

audio_reducido_v2 = nr.reduce_noise(
    y=audio_base,
    sr=frecuencia_procesamiento,
    stationary=False,
    prop_decrease=0.80,
    time_constant_s=1.0,
    freq_mask_smooth_hz=300,
    time_mask_smooth_ms=80
)

print("REDUCCIÓN NO ESTACIONARIA COMPLETADA")
print("-" * 50)
print("Modo utilizado: ruido variable")
print("Intensidad de reducción: 0.80")
print("Muestras procesadas:", len(audio_reducido_v2))

In [ ]:
print("Audio antes de la nueva reducción:")
display(Audio(audio_base, rate=frecuencia_procesamiento))

print("Audio después de la nueva reducción:")
display(Audio(audio_reducido_v2, rate=frecuencia_procesamiento))

In [ ]:
# Puerta de ruido suave para atenuar el sonido durante las pausas

def aplicar_puerta_ruido(
    senal,
    sr,
    frame_length=2048,
    hop_length=512,
    percentil_umbral=30,
    atenuacion=0.15
):
    # Calcular energía RMS por fragmentos
    rms = librosa.feature.rms(
        y=senal,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    # Umbral basado en la distribución de energía
    umbral = np.percentile(
        rms,
        percentil_umbral
    )

    # Crear una ganancia por cada fragmento
    ganancia_frames = np.where(
        rms < umbral,
        atenuacion,
        1.0
    )

    # Suavizar los cambios para evitar cortes bruscos
    ventana_suavizado = 9

    kernel = (
        np.ones(ventana_suavizado)
        / ventana_suavizado
    )

    ganancia_frames = np.convolve(
        ganancia_frames,
        kernel,
        mode="same"
    )

    # Convertir la ganancia de fragmentos al tamaño de la señal
    posiciones_frames = librosa.frames_to_samples(
        np.arange(len(ganancia_frames)),
        hop_length=hop_length
    )

    posiciones_frames = np.clip(
        posiciones_frames,
        0,
        len(senal) - 1
    )

    ganancia_muestras = np.interp(
        np.arange(len(senal)),
        posiciones_frames,
        ganancia_frames
    )

    senal_atenuada = senal * ganancia_muestras

    return senal_atenuada, umbral


audio_con_puerta, umbral_puerta = aplicar_puerta_ruido(
    audio_reducido_v2,
    frecuencia_procesamiento,
    percentil_umbral=30,
    atenuacion=0.15
)

print("PUERTA DE RUIDO APLICADA")
print("-" * 45)
print("Umbral RMS:", round(float(umbral_puerta), 8))
print("Atenuación durante las pausas: 85 %")
print("Se conserva durante las pausas: 15 %")


In [ ]:
print("1. AUDIO DESPUÉS DE RECORTAR EL INICIO")
display(Audio(audio_base, rate=frecuencia_procesamiento))

print("2. AUDIO CON REDUCCIÓN NO ESTACIONARIA")
display(Audio(audio_reducido_v2, rate=frecuencia_procesamiento))

print("3. AUDIO CON REDUCCIÓN Y PUERTA DE RUIDO")
display(Audio(audio_con_puerta, rate=frecuencia_procesamiento))

In [ ]:
# Filtro paso alto para reducir vibraciones y ruido grave

def aplicar_filtro_paso_alto(
    senal,
    sr,
    frecuencia_corte=80,
    orden=4
):
    sos = butter(
        orden,
        frecuencia_corte,
        btype="highpass",
        fs=sr,
        output="sos"
    )

    return sosfiltfilt(sos, senal)


audio_filtrado = aplicar_filtro_paso_alto(
    audio_con_puerta,
    frecuencia_procesamiento,
    frecuencia_corte=80,
    orden=4
)

print("FILTRO PASO ALTO APLICADO")
print("-" * 40)
print("Frecuencia de corte: 80 Hz")
print("Orden del filtro: 4")

display(Audio(audio_filtrado, rate=frecuencia_procesamiento))

In [ ]:
# Normalización final

amplitud_antes = np.max(
    np.abs(audio_filtrado)
)

if amplitud_antes > 0:
    audio_normalizado = (
        0.95 * audio_filtrado / amplitud_antes
    )
else:
    audio_normalizado = audio_filtrado.copy()

print("NORMALIZACIÓN FINAL")
print("-" * 40)
print(
    "Amplitud antes:",
    round(float(amplitud_antes), 6)
)
print(
    "Amplitud después:",
    round(
        float(
            np.max(np.abs(audio_normalizado))
        ),
        6
    )
)

display(
    Audio(
        audio_normalizado,
        rate=frecuencia_procesamiento
    )
)

La comparación auditiva permitió identificar una reducción considerable de los ruidos de fondo, especialmente durante las pausas del discurso. La voz se conservó comprensible y el mensaje permaneció completo después del procesamiento. Sin embargo, se mantuvo parcialmente un ruido fuerte que coincidía temporalmente con la voz al comienzo de la grabación. Esta interferencia no fue eliminada completamente, debido a que compartía el mismo intervalo temporal con la señal de voz y una reducción más agresiva podía afectar la inteligibilidad y naturalidad del mensaje.

# 6. Análisis de resultados

En esta sección se compararon la señal original y la señal procesada mediante formas de onda, espectrogramas y métricas de energía. La comparación permitió identificar los cambios producidos por el recorte inicial, la reducción no estacionaria de ruido, la puerta de ruido, el filtrado paso alto y la normalización. Además, se realizó una evaluación auditiva para verificar la conservación de la voz y la disminución de las interferencias.

In [ ]:
# Comparación de las formas de onda

fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 8)
)

librosa.display.waveshow(
    audio_remuestreado,
    sr=frecuencia_procesamiento,
    ax=axes[0],
    color="steelblue"
)

axes[0].set_title("Forma de onda del audio original")
axes[0].set_xlabel("Tiempo (segundos)")
axes[0].set_ylabel("Amplitud")
axes[0].grid(alpha=0.2)

librosa.display.waveshow(
    audio_normalizado,
    sr=frecuencia_procesamiento,
    ax=axes[1],
    color="seagreen"
)

axes[1].set_title("Forma de onda del audio procesado")
axes[1].set_xlabel("Tiempo (segundos)")
axes[1].set_ylabel("Amplitud")
axes[1].grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    "comparacion_formas_onda.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Espectrograma del audio original

stft_original = librosa.stft(audio_remuestreado)

espectrograma_original = librosa.amplitude_to_db(
    np.abs(stft_original),
    ref=np.max
)

# Espectrograma del audio procesado

stft_procesado = librosa.stft(audio_normalizado)

espectrograma_procesado = librosa.amplitude_to_db(
    np.abs(stft_procesado),
    ref=np.max
)

# Comparación gráfica

fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 10)
)

imagen_original = librosa.display.specshow(
    espectrograma_original,
    sr=frecuencia_procesamiento,
    x_axis="time",
    y_axis="hz",
    cmap="magma",
    ax=axes[0]
)

axes[0].set_title("Espectrograma del audio original")
axes[0].set_ylim(0, 8000)

fig.colorbar(
    imagen_original,
    ax=axes[0],
    format="%+2.0f dB"
)

imagen_procesada = librosa.display.specshow(
    espectrograma_procesado,
    sr=frecuencia_procesamiento,
    x_axis="time",
    y_axis="hz",
    cmap="magma",
    ax=axes[1]
)

axes[1].set_title("Espectrograma del audio procesado")
axes[1].set_ylim(0, 8000)

fig.colorbar(
    imagen_procesada,
    ax=axes[1],
    format="%+2.0f dB"
)

plt.tight_layout()

plt.savefig(
    "comparacion_espectrogramas.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Funciones para calcular las métricas

def calcular_rms(senal):
    return np.sqrt(
        np.mean(senal ** 2)
    )


def convertir_a_dbfs(valor):
    return 20 * np.log10(
        valor + 1e-12
    )


# Métricas del audio original

rms_original = calcular_rms(
    audio_remuestreado
)

dbfs_original = convertir_a_dbfs(
    rms_original
)

pico_original = np.max(
    np.abs(audio_remuestreado)
)

duracion_original = (
    len(audio_remuestreado)
    / frecuencia_procesamiento
)


# Métricas del audio procesado

rms_procesado = calcular_rms(
    audio_normalizado
)

dbfs_procesado = convertir_a_dbfs(
    rms_procesado
)

pico_procesado = np.max(
    np.abs(audio_normalizado)
)

duracion_procesada = (
    len(audio_normalizado)
    / frecuencia_procesamiento
)


print("MÉTRICAS DEL TRATAMIENTO")
print("-" * 50)
print(
    "Duración original:",
    round(duracion_original, 2),
    "segundos"
)
print(
    "Duración procesada:",
    round(duracion_procesada, 2),
    "segundos"
)
print(
    "RMS original:",
    round(float(rms_original), 6)
)
print(
    "RMS procesado:",
    round(float(rms_procesado), 6)
)
print(
    "Nivel RMS original:",
    round(float(dbfs_original), 2),
    "dBFS"
)
print(
    "Nivel RMS procesado:",
    round(float(dbfs_procesado), 2),
    "dBFS"
)
print(
    "Pico original:",
    round(float(pico_original), 6)
)
print(
    "Pico procesado:",
    round(float(pico_procesado), 6)
)

In [ ]:
import pandas as pd

tabla_resultados = pd.DataFrame({
    "Métrica": [
        "Duración",
        "Frecuencia de muestreo",
        "Energía RMS",
        "Nivel RMS",
        "Amplitud máxima"
    ],
    "Audio original": [
        f"{duracion_original:.2f} s",
        f"{frecuencia_procesamiento} Hz",
        f"{rms_original:.6f}",
        f"{dbfs_original:.2f} dBFS",
        f"{pico_original:.6f}"
    ],
    "Audio procesado": [
        f"{duracion_procesada:.2f} s",
        f"{frecuencia_procesamiento} Hz",
        f"{rms_procesado:.6f}",
        f"{dbfs_procesado:.2f} dBFS",
        f"{pico_procesado:.6f}"
    ]
})

display(tabla_resultados)

tabla_resultados.to_csv(
    "resultados_tratamiento_audio.csv",
    index=False
)

Interpretación de los resultados

La grabación original presentó una duración de 29,99 segundos, mientras que la señal procesada alcanzó una duración de 28,99 segundos. La diferencia de 1 segundo se relaciona con la eliminación del ruido fuerte ubicado antes del inicio de la voz. Tanto el audio original como el procesado utilizaron una frecuencia de muestreo de 22.050 Hz, lo que permitió mantener condiciones equivalentes para su comparación.

El audio original presentó una energía RMS de 0,072460 y un nivel RMS de −22,80 dBFS. Después del tratamiento, la señal procesada alcanzó una energía RMS de 0,081529 y un nivel RMS de −21,77 dBFS. El incremento de estos valores no indica un aumento del ruido, debido a que durante la etapa final se normalizó la señal para mejorar su nivel de reproducción. Como resultado de esta normalización, la amplitud máxima pasó de 0,694983 en el audio original a 0,950000 en el audio procesado, correspondiente al valor objetivo definido en el procedimiento.

La comparación auditiva permitió identificar una reducción considerable de los ruidos de fondo, especialmente durante las pausas del discurso. La aplicación de la reducción no estacionaria y de la puerta de ruido disminuyó las interferencias de baja intensidad, mientras que el filtro paso alto atenuó componentes graves inferiores a 80 Hz. La voz se conservó comprensible y el mensaje hablado permaneció completo después del procesamiento.

En las formas de onda se observó una modificación de la amplitud y una disminución de la duración total debido al recorte del segmento inicial. Por su parte, los espectrogramas permitieron identificar una menor presencia de energía durante algunas pausas y en las frecuencias más bajas. No obstante, permaneció parcialmente una interferencia fuerte superpuesta con la voz al comienzo de la grabación. Esta interferencia no fue eliminada completamente porque coincidía temporalmente con la señal de voz y un tratamiento más agresivo podía producir distorsión, pérdida de información o disminución de la inteligibilidad del mensaje.

En conjunto, los resultados muestran que el procedimiento aplicado mejoró las condiciones generales de la grabación, redujo los ruidos presentes durante las pausas y mantuvo la información principal de la voz. La valoración final se realizó combinando las métricas numéricas, las formas de onda, los espectrogramas y la comparación auditiva, debido a que ninguna de estas evidencias, utilizada de manera aislada, permite determinar completamente la calidad del tratamiento.

# 7. Exportación del audio procesado

En esta sección se exportó la señal obtenida después de aplicar el recorte inicial, la reducción no estacionaria de ruido, la puerta de ruido, el filtro paso alto y la normalización. El resultado fue almacenado en formato WAV con una frecuencia de muestreo de 22.050 Hz. También se descargaron las gráficas, la tabla comparativa y el cuaderno de Google Colab como evidencias del procedimiento desarrollado.

In [ ]:
# Exportación del audio procesado

nombre_audio_salida = "audio_procesado_Dairo_Contreras.wav"

sf.write(
    nombre_audio_salida,
    audio_normalizado,
    frecuencia_procesamiento
)

print("EXPORTACIÓN DEL AUDIO PROCESADO")
print("-" * 50)
print("Nombre del archivo:", nombre_audio_salida)
print("Formato: WAV")
print("Frecuencia de muestreo:", frecuencia_procesamiento, "Hz")
print(
    "Duración:",
    round(
        len(audio_normalizado) / frecuencia_procesamiento,
        2
    ),
    "segundos"
)
print("Canales: 1, señal mono")
print("Archivo guardado correctamente")


In [ ]:
# Verificación del archivo exportado

audio_verificacion, sr_verificacion = sf.read(
    nombre_audio_salida
)

print("VERIFICACIÓN DEL ARCHIVO")
print("-" * 45)
print("Archivo leído correctamente")
print("Cantidad de muestras:", len(audio_verificacion))
print("Frecuencia de muestreo:", sr_verificacion, "Hz")
print(
    "Duración:",
    round(
        len(audio_verificacion) / sr_verificacion,
        2
    ),
    "segundos"
)
print(
    "Amplitud máxima:",
    round(
        float(np.max(np.abs(audio_verificacion))),
        6
    )
)

In [ ]:
print("Reproducción del archivo WAV exportado:")

display(
    Audio(
        audio_verificacion,
        rate=sr_verificacion
    )
)

In [ ]:
files.download(nombre_audio_salida)

El resultado final fue exportado con el nombre audio_procesado_Dairo_Contreras.wav, en formato WAV, señal mono y frecuencia de muestreo de 22.050 Hz. El archivo presentó una duración de 28,99 segundos y una amplitud máxima normalizada de 0,95. Posteriormente, el archivo fue cargado y reproducido nuevamente para verificar su integridad. La comprobación confirmó que el audio podía reproducirse correctamente y que conservaba el mensaje hablado después del tratamiento.

También se exportaron las gráficas de las formas de onda, los espectrogramas y la tabla de resultados. Estos archivos se conservaron como evidencias del procedimiento.